# Finish control_strong seeds 8–11 (Kaggle, resume from checkpoints)

Resumes SimCLR training for the four remaining **control_strong** encoders
(`control_strong_seed8 … seed11`, Shapes3D) from checkpoints you upload, and
trains each to the 50-epoch cut. Nothing else in the grid is touched.

## One-time setup
1. **Upload your checkpoints as a Kaggle Dataset**, one file per seed. Any of
   these layouts is recognized (section 5 auto-detects the seed):
   - foldered: `control_strong_seed8/last_ckpt.pt`, `…seed9/last_ckpt.pt`, …
   - seed-suffixed flat files: `last_ckpt_seed8.pt` (or `checkpoint_seed8.pt`), …
   The seed number **must** appear in the path or filename — that is how each
   file is routed to the right run.
2. **Settings → Accelerator → `GPU T4 x2`** (or single T4 — both work; x2 trains
   two seeds at once).
3. **Settings → Internet → On** (clone + install + Shapes3D download).
4. **Add Input →** your checkpoint dataset (so it mounts under `/kaggle/input`).

## Run order
Sections 1–5 top to bottom (clone, install, data, restore checkpoints), then
**section 6** (train). Section 6 is idempotent + resumable: it skips a seed once
its `backbone.pt` exists and resumes a partial seed from its `last_ckpt.pt`, so
**re-run section 6 after a session ends** to continue. **Save Version** every few
hours so `/kaggle/working` persists (Kaggle sessions cap ~12 h).

Wall-clock: ~5 GPU-h for a full 50-epoch encoder from scratch; less from a
partial checkpoint. Four seeds ÷ 2 GPUs ≈ a single session if the checkpoints are
already well underway.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download Shapes3D + build the image cache
control_strong is a Shapes3D cell, so only Shapes3D is needed. `--build-cache`
decompresses once into an uncompressed memmap the trainers mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore your uploaded checkpoints → `results/encoders/control_strong_seed<N>/`
Auto-discovers `*.pt` under `/kaggle/input`, keeps only **seeds 8–11**, and copies
each into the run dir the trainer resumes from. A `last_ckpt`/`checkpoint` file
→ `last_ckpt.pt` (section 6 resumes it); a `backbone` file → `backbone.pt` (already
complete, section 6 skips it). The seed is read from the path/filename; files
whose seed can't be determined, or whose seed is outside 8–11, are ignored and
listed so you can rename + re-add them.

In [ ]:
import re, shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")

CONDITION = "control_strong"   # this notebook's cell
SEEDS = {8, 9, 10, 11}          # the four we are finishing

def _seed(p):
    m = re.search(rf"{CONDITION}_seed(\d+)", p.as_posix())   # full run id in path
    if m:
        return int(m.group(1))
    m = re.search(r"seed[_-]?(\d+)", p.name)                 # seed<N>/seed_<N> in name
    return int(m.group(1)) if m else None

def _target(name):
    n = name.lower()
    if "backbone" in n:
        return "backbone.pt"                                 # completed encoder
    if "ckpt" in n or "checkpoint" in n or "last" in n:
        return "last_ckpt.pt"                                # resume file
    return "last_ckpt.pt"                                    # default: treat as resume

found, ignored = {}, []
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    s = _seed(p)
    if s in SEEDS:
        found.setdefault((s, _target(p.name)), p)           # first match per (seed, kind) wins
    else:
        ignored.append((p, s))

if not found:
    print("No matching checkpoints under /kaggle/input.")
    print("Add Input -> your checkpoint dataset, and make sure each file's name/path")
    print("contains its seed (e.g. control_strong_seed8/last_ckpt.pt or last_ckpt_seed8.pt).")
for (s, tgt), src in sorted(found.items()):
    dst = ENC / f"{CONDITION}_seed{s}" / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"OK seed{s}: {src.name} -> {dst.relative_to(REPO)} ({dst.stat().st_size >> 20} MB)")

if ignored:
    print("\nIgnored (seed missing or not in 8-11):")
    for p, s in ignored:
        print(f"  seed={s}  {p}")

## 6. Resume training — control_strong seeds 8–11
Launches up to one `train_simclr` per GPU, each pinned via `CUDA_VISIBLE_DEVICES`,
resuming from the restored `last_ckpt.pt` and running to 50 epochs. Per-seed logs
in `results/sweep_logs/`. Idempotent + resumable — **re-run this cell to continue**
after a session ends; done seeds are skipped.

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)

CONFIG = "configs/run/control_strong.yaml"
SEEDS = [8, 9, 10, 11]
N_GPUS = max(1, torch.cuda.device_count())

def run_id(seed): return f"control_strong_seed{seed}"
def is_done(seed): return (ENC / run_id(seed) / "backbone.pt").exists()

def _launch(seed, gpu):
    rid = run_id(seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", CONFIG,
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run(n_gpus=N_GPUS, poll=15):
    subprocess.run(["pkill", "-9", "-f", "src.encoders.train_simclr"])  # reap orphans
    time.sleep(2)
    queue = [s for s in SEEDS if not is_done(s)]
    print(f"{len(SEEDS)} seeds | {len(SEEDS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    seed = queue.pop(0)
                    if is_done(seed):
                        continue
                    slots[g] = _launch(seed, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    seed = int(s["rid"].split("seed")[1])
                    ok = is_done(seed)
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("done — all four seeds have a backbone.pt")

run()

## 7. Monitor
Re-run any time while section 6 is training.

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 8. Progress + final diagnostics
Per-seed state and, once done, the collapse diagnostics for each encoder.

In [ ]:
import json as _json

for seed in [8, 9, 10, 11]:
    rid = f"control_strong_seed{seed}"; d = ENC / rid
    if (d / "backbone.pt").exists():
        m = _json.loads((d / "metrics.json").read_text()); diag = m.get("diagnostics", {})
        print(f"{rid}: DONE loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
              f"feat_std={diag.get('feat_std'):.4f} eff_rank={diag.get('eff_rank'):.1f} "
              f"align={diag.get('alignment'):.3f} unif={diag.get('uniformity'):.3f}")
    elif (d / "last_ckpt.pt").exists():
        last = None
        for line in (d / "train_log.jsonl").read_text().splitlines():
            try: last = _json.loads(line).get("epoch", last)
            except Exception: pass
        print(f"{rid}: partial @ep{last}")
    else:
        print(f"{rid}: todo (no checkpoint restored)")

## 9. Save the finished encoders
**Save Version (Commit)** to persist `/kaggle/working`. The four `backbone.pt`
under `results/encoders/control_strong_seed{8,9,10,11}/` are the deliverable —
feed them back into the full sweep notebook's probing step (section 10 there)
once all 12 control_strong seeds exist.